In [2]:
import os
import sys
import pandas as pd
import numpy as np
import openpyxl
import sys

In [3]:
df = pd.read_excel("../Datos/Originales/Prestamos_Data_Alumnos_v3.xlsx")

df = df.dropna(subset=["Prima"]).copy()

df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,S97R7X,18,16000,5000,397,19,1,8.06,48,0.10,Escolar,Autónomo,Soltero,1,0,Automóvil,0,0,50.12
1,T3ZE0N,69,72673,32340,784,320,2,15.04,48,0.12,Doctorado,Tiempo parcial,Casado,0,1,Educación,0,0,187.15
2,RLGTBY,50,62116,37278,486,217,3,21.96,12,0.55,Grado Universitario,Tiempo parcial,Casado,1,0,Automóvil,1,1,800.00
3,BZ86CV,64,59846,19784,308,340,1,24.26,12,0.35,Grado Universitario,Desempleado,Divorciado,1,0,Negocios,1,0,275.82
4,5OD75M,62,28413,13751,412,476,2,5.73,36,0.14,Escolar,Autónomo,Casado,0,0,Negocios,0,0,100.07


In [4]:
df = df[df["Proposito"] == "Vivienda"]

In [5]:
# --- Duración en meses -> años ---
df["Duracion_anios"] = df["Duracion"] / 12

# Ingresos totales estimados durante el préstamo
df["Ingresos_totales_prestamo"] = df["Ingresos"] * df["Duracion_anios"]

# Casos inválidos: ingresos durante el préstamo < monto inicial
mask_invalidos = df["Ingresos_totales_prestamo"] < df["Monto_Inicial"]

df_ingresos_invalidos = df[mask_invalidos].copy()

print("Casos inválidos detectados:", df_ingresos_invalidos.shape[0])
print("Porcentaje inválidos:       ", round(df_ingresos_invalidos.shape[0] / df.shape[0] * 100, 2), "%")

# Ver ejemplos
df_ingresos_invalidos[["ID", "Ingresos", "Duracion", "Duracion_anios", "Ingresos_totales_prestamo", "Monto_Inicial"]].head(10)


Casos inválidos detectados: 20999
Porcentaje inválidos:        40.94 %


,ID,Ingresos,Duracion,Duracion_anios,Ingresos_totales_prestamo,Monto_Inicial
11,ABUXPP,77019,36,3.0,231057.0,235524
19,A36W02,17373,24,2.0,34746.0,40000
23,YCY979,43160,60,5.0,215800.0,354632
33,OMGQXP,59152,36,3.0,177456.0,187320
37,6E4EPG,44284,36,3.0,132852.0,236709
40,4HWVAO,40869,24,2.0,81738.0,114651
41,O8N21E,44633,12,1.0,44633.0,95942
47,34HV1K,51135,24,2.0,102270.0,145563
52,FQLDPX,66775,24,2.0,133550.0,263909
63,CLEYJQ,42187,12,1.0,42187.0,157558


In [6]:
df = df[~mask_invalidos].copy()

In [7]:
# -------------------------------
# Eliminar todos los Prima = 800
# y reconstruir el rango 760–800 sin ellos
# -------------------------------

# 1) Extraemos el rango alto
df_rango_alto = df[(df["Prima"] >= 760) & (df["Prima"] <= 800)].copy()

# 2) Nos quedamos SOLO con los que NO son 800
df_rango_sin_800 = df_rango_alto[df_rango_alto["Prima"] != 800].copy()

print("En rango 760–800, NO 800:", len(df_rango_sin_800))
print("En rango 760–800, 800 eliminados:", (df_rango_alto["Prima"] == 800).sum())

# 3) Quitamos el rango original del df
df = df[~((df["Prima"] >= 760) & (df["Prima"] <= 800))].copy()

# 4) Añadimos el rango ya limpiado (sin 800)
df = pd.concat([df, df_rango_sin_800], ignore_index=True)

print("Nuevo tamaño df:", df.shape)
print("Distribución del rango alto (>=760):")
print(df[df["Prima"] >= 760]["Prima"].value_counts().head(20))


En rango 760–800, NO 800: 736
En rango 760–800, 800 eliminados: 23648
Nuevo tamaño df: (6639, 21)
Distribución del rango alto (>=760):
Prima
785.05    4
795.29    3
785.54    3
765.89    3
773.98    3
764.79    2
783.00    2
764.12    2
763.64    2
788.76    2
779.10    2
799.88    2
799.62    2
765.39    2
774.75    2
766.21    2
786.07    2
769.59    2
797.24    2
765.83    2
Name: count, dtype: int64


In [8]:
df.duplicated(subset="ID").sum()

#duplicados

np.int64(0)

In [9]:
df.isna().sum()

#nulos

ID                           0
Edad                         0
Ingresos                     0
Monto_Inicial                0
Scoring_Crediticio           0
Meses_Empleo                 0
Num_Creditos                 0
Ratio_Interes                0
Duracion                     0
Ratio_Deuda_Ingresos         0
Estudios                     0
Tipo_Jornada_Laboral         0
Estado_Civil                 0
Posesion_Hipoteca            0
Personas_Cargo               0
Proposito                    0
Fiador                       0
Impago                       0
Prima                        0
Duracion_anios               0
Ingresos_totales_prestamo    0
dtype: int64

In [10]:
df["Meses_Maximos"] = (df["Edad"] - 16) * 12
df_invalidos = df[df["Meses_Empleo"] > df["Meses_Maximos"]]

In [11]:
df = df[df["Meses_Empleo"] <= df["Meses_Maximos"]]
df = df.drop(columns="Meses_Maximos")

## Se eliminaron registros con cosas no lógicas en la variable Meses_Empleo, ya que implicaban experiencia laboral previa 
# a la edad legal mínima (16 años).

In [12]:
df = df[(df["Edad"] >= 16) & (df["Edad"] <= 100)]

# edades, no menor a 16 y mayores a 100

In [13]:
df["Tipo_Jornada_Laboral"].value_counts()

Tipo_Jornada_Laboral
Autónomo            1695
Jornada completa    1668
Tiempo parcial      1653
Desempleado         1623
Name: count, dtype: int64

In [14]:
df["Tipo_Jornada_Laboral"] = (
    df["Tipo_Jornada_Laboral"]
    .str.strip()
    .str.lower()
)

df["Tipo_Jornada_Laboral"] = df["Tipo_Jornada_Laboral"].replace({
    "autonomo": "autónomo"
})

df["Tipo_Jornada_Laboral"].value_counts()

## por si hay alguna palabra mal, en plan espacios masculino/femenino

Tipo_Jornada_Laboral
autónomo            1695
jornada completa    1668
tiempo parcial      1653
desempleado         1623
Name: count, dtype: int64

In [15]:
df[~df["Fiador"].isin([0,1])]

## tiene que ser binario

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima,Duracion_anios,Ingresos_totales_prestamo


In [16]:
df = df[df["Ingresos"] > 0]

## ingresos siempre positivos, no se pueden negativos

In [17]:
df = df[df["Monto_Inicial"] > 0]

## monto inicial imposible negativo

In [18]:
df = df[(df["Ratio_Deuda_Ingresos"] >= 0) & (df["Ratio_Deuda_Ingresos"] <= 1)]

# no puedes menos del 0 y no puedes deber mas de 1

In [19]:
df = df[(df["Ratio_Interes"] > 0) & (df["Ratio_Interes"] <= 100)]

# no se puede menos de 0 ni mas de 100

In [20]:
df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6639 entries, 0 to 6638
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         6639 non-null   object 
 1   Edad                       6639 non-null   int64  
 2   Ingresos                   6639 non-null   int64  
 3   Monto_Inicial              6639 non-null   int64  
 4   Scoring_Crediticio         6639 non-null   int64  
 5   Meses_Empleo               6639 non-null   int64  
 6   Num_Creditos               6639 non-null   int64  
 7   Ratio_Interes              6639 non-null   float64
 8   Duracion                   6639 non-null   int64  
 9   Ratio_Deuda_Ingresos       6639 non-null   float64
 10  Estudios                   6639 non-null   object 
 11  Tipo_Jornada_Laboral       6639 non-null   object 
 12  Estado_Civil               6639 non-null   object 
 13  Posesion_Hipoteca          6639 non-null   int64

In [21]:
condiciones_invalidas = (
    ((df["Edad"] < 19) & (df["Estudios"] == "grado")) |
    ((df["Edad"] < 20) & (df["Estudios"] == "máster")) |
    ((df["Edad"] < 25) & (df["Estudios"] == "doctorado"))
)

df_estudios_invalidos = df[condiciones_invalidas]
len(df_estudios_invalidos)

0

In [22]:
df.shape

(6639, 21)

In [23]:
df.to_excel(
    "../Datos/Limpios/información_préstamos_limpio.xlsx",
    index=False
)
